# Compression of nii files
Because Kaggle unzipped my zipped files.

In [11]:
import os

# Define the path to the directory you want to remove
directory_path = '/kaggle/working/gz_CN_TimePoints_3'

# Remove the directory and all its contents
os.system(f'rm -r {directory_path}')

# Verify the directory has been removed
if not os.path.exists(directory_path):
    print(f"The directory '{directory_path}' has been successfully removed.")
else:
    print(f"Failed to remove the directory '{directory_path}'.")
    

sh: 0: getcwd() failed: No such file or directory


The directory '/kaggle/working/gz_CN_TimePoints_3' has been successfully removed.


In [6]:
import os
import nibabel as nib
import gzip
import shutil

In [7]:
# Define the directory containing the .nii files
input_directory = '/kaggle/input/articleimage'
output_directory = '/kaggle/working/processedFolder'

# Create the output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Iterate through all files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith('.nii'):
        file_path = os.path.join(input_directory, filename)
        
        # Load the .nii file using nibabel
        img = nib.load(file_path)
        
        # Create a .nii.gz filename
        compressed_filename = filename + '.gz'
        compressed_file_path = os.path.join(output_directory, compressed_filename)
        
        # Save the .nii file as .nii.gz using nibabel
        nib.save(img, compressed_file_path)

print("Compression completed.")


Compression completed.


In [3]:
                            ### check for remaining files ###
import os
import shutil

def copy_nii_gz_files(src_folder, dest_folder):
    """
    Copies all .nii.gz files from the source folder to the destination folder.

    :param src_folder: str, path to the source folder containing .nii.gz files
    :param dest_folder: str, path to the destination folder where the files will be copied
    """
    
    # Iterate through all files in the source folder
    for filename in os.listdir(src_folder):
        # Check if the file has a .nii.gz extension
        if filename.endswith('.nii.gz'):
            src_file_path = os.path.join(src_folder, filename)
            dest_file_path = os.path.join(dest_folder, filename)
            
            # Copy the file
            shutil.copy(src_file_path, dest_file_path)
            print(f"File copied from {src_file_path} to {dest_file_path}")

# Example usage:
src_folder = '/kaggle/input/cn-corrected-3'
dest_folder = '/kaggle/working/gz_CN_TimePoints_3'

copy_nii_gz_files(src_folder, dest_folder)


# N4 Bias Field Correction

In [8]:
pip install SimpleITK

Note: you may need to restart the kernel to use updated packages.


In [9]:
# helper functions
import matplotlib.pyplot as plt

from ipywidgets import interact
import numpy as np
import SimpleITK as sitk
import cv2

def explore_3D_array(arr: np.ndarray, cmap: str = 'gray'):
    """
  Given a 3D array with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D array.
  The purpose of this function to visual inspect the 2D arrays in the image.

  Args:
    arr : 3D array with shape (Z,X,Y) that represents the volume of a MRI image
    cmap : Which color map use to plot the slices in matplotlib.pyplot
  """

    def fn(SLICE):
        plt.figure(figsize=(7,7))
        plt.imshow(arr[SLICE, :, :], cmap=cmap)

    interact(fn, SLICE=(0, arr.shape[0]-1))


def explore_3D_array_comparison(arr_before: np.ndarray, arr_after: np.ndarray, cmap: str = 'gray'):
    """"
  Given two 3D arrays with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D arrays.
  The purpose of this function to visual compare the 2D arrays after some transformation.

  Args:
    arr_before : 3D array with shape (Z,X,Y) that represents the volume of a MRI image, before any transform
    arr_after : 3D array with shape (Z,X,Y) that represents the volume of a MRI image, after some transform
    cmap : Which color map use to plot the slices in matplotlib.pyplot
  """

    assert arr_after.shape == arr_before.shape

    def fn(SLICE):
        fig, (ax1, ax2) = plt.subplots(1, 2, sharex='col', sharey='row', figsize=(10,10))

        ax1.set_title('Before', fontsize=15)
        ax1.imshow(arr_before[SLICE, :, :], cmap=cmap)

        ax2.set_title('After', fontsize=15)
        ax2.imshow(arr_after[SLICE, :, :], cmap=cmap)

        plt.tight_layout()

    interact(fn, SLICE=(0, arr_before.shape[0]-1))


def show_sitk_img_info(img: sitk.Image):
    """
  Given a sitk.Image instance prints the information about the MRI image contained.

  Args:
    img : instance of the sitk.Image to check out
  """
    pixel_type = img.GetPixelIDTypeAsString()
    origin = img.GetOrigin()
    dimensions = img.GetSize()
    spacing = img.GetSpacing()
    direction = img.GetDirection()

    info = {'Pixel Type' : pixel_type, 'Dimensions': dimensions, 'Spacing': spacing, 'Origin': origin,  'Direction' : direction}
    for k,v in info.items():
        print(f' {k} : {v}')


def add_suffix_to_filename(filename: str, suffix:str) -> str:
    """
  Takes a NIfTI filename and appends a suffix.

  Args:
      filename : NIfTI filename
      suffix : suffix to append

  Returns:
      str : filename after append the suffix
  """
    if filename.endswith('.nii'):
        result = filename.replace('.nii', f'_{suffix}.nii')
        return result
    elif filename.endswith('.nii.gz'):
        result = filename.replace('.nii.gz', f'_{suffix}.nii.gz')
        return result
    else:
        raise RuntimeError('filename with unknown extension')


def rescale_linear(array: np.ndarray, new_min: int, new_max: int):
    """Rescale an array linearly."""
    minimum, maximum = np.min(array), np.max(array)
    m = (new_max - new_min) / (maximum - minimum)
    b = new_min - m * minimum
    return m * array + b


def explore_3D_array_with_mask_contour(arr: np.ndarray, mask: np.ndarray, thickness: int = 1):
    """
  Given a 3D array with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D array. The binary
  mask provided will be used to overlay contours of the region of interest over the
  array. The purpose of this function is to visual inspect the region delimited by the mask.

  Args:
    arr : 3D array with shape (Z,X,Y) that represents the volume of a MRI image
    mask : binary mask to obtain the region of interest
  """
    assert arr.shape == mask.shape

    _arr = rescale_linear(arr,0,1)
    _mask = rescale_linear(mask,0,1)
    _mask = _mask.astype(np.uint8)

    def fn(SLICE):
        arr_rgb = cv2.cvtColor(_arr[SLICE, :, :], cv2.COLOR_GRAY2RGB)
        contours, _ = cv2.findContours(_mask[SLICE, :, :], cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        arr_with_contours = cv2.drawContours(arr_rgb, contours, -1, (0,1,0), thickness)

        plt.figure(figsize=(7,7))
        plt.imshow(arr_with_contours)

    interact(fn, SLICE=(0, arr.shape[0]-1))

In [ ]:
# pip install ants

In [10]:
%matplotlib inline
# matplotlib will be displayed inline in the notebook

import os
import sys

# appending the path to include our custom helper
# import ants
import SimpleITK as sitk

In [ ]:
# !unzip '/content/drive/MyDrive/Thesis/Data/CN/CN_baseline_cls9_PR.zip' -d '/content/drive/MyDrive/Thesis/Data/CN'

In [11]:
BASE_DIR = os.path.dirname("/kaggle/working/")
print(f'project folder = {BASE_DIR}')

# Define the directory path
directory_path = '/kaggle/working/processedFolder/'
# Iterate through files in the directory and add filenames with extensions to raw_examples list

raw_examples = []
all_files = os.listdir(directory_path)
all_files.sort()
choosen_files = all_files[:100]

# for filename in os.listdir(directory_path):
for filename in choosen_files:
    # Check if the path refers to a file (not a subdirectory)
    if os.path.isfile(os.path.join(directory_path,filename)):
        raw_examples.append(filename)

# Display the updated raw_examples list
print(len(raw_examples))
print(raw_examples)

project folder = /kaggle/working
1
['002_S_0816_002_S_0816.nii.gz']


In [12]:
# loop through the entire dataset to do N4 bias field correction
OUTPUT_DIR = os.path.dirname("/kaggle/working/")
for volume in raw_examples:
    raw_img_path = os.path.join(BASE_DIR, 'processedFolder', volume)
    raw_img_sitk = sitk.ReadImage(raw_img_path, sitk.sitkFloat32)
    raw_img_sitk = sitk.DICOMOrient(raw_img_sitk,'RPS')

    # Create head mask
    transformed = sitk.RescaleIntensity(raw_img_sitk, 0, 255)

    #transformed = sitk.TriangleThreshold(transformed, 0, 1)
    transformed = sitk.LiThreshold(transformed,0,1)

    head_mask = transformed
    # Bias Correction
    shrinkFactor = 4
    inputImage = raw_img_sitk

    inputImage = sitk.Shrink( raw_img_sitk, [ shrinkFactor ] * inputImage.GetDimension() )
    maskImage = sitk.Shrink( head_mask, [ shrinkFactor ] * inputImage.GetDimension() )

    bias_corrector = sitk.N4BiasFieldCorrectionImageFilter()

    corrected = bias_corrector.Execute(inputImage, maskImage)
    # Get image corrected
    log_bias_field = bias_corrector.GetLogBiasFieldAsImage(raw_img_sitk)
    corrected_image_full_resolution = raw_img_sitk / sitk.Exp( log_bias_field )
    # Save the image
    out_folder =  os.path.join(OUTPUT_DIR, 'BiasFieldCorrected')
    os.makedirs(out_folder, exist_ok=True) # create folder if not exists

    out_filename = add_suffix_to_filename(volume, suffix='biasFieldCorrected')
    out_path = os.path.join(out_folder, out_filename)

    sitk.WriteImage(corrected_image_full_resolution, out_path)


In [13]:
for filename in os.listdir(out_folder):
    # Check if the path refers to a file (not a subdirectory)
    if os.path.isfile(os.path.join(OUTPUT_DIR,filename)):
        raw_examples.append(filename)

# Display the updated raw_examples list
print(len(raw_examples))

1


# HD-BET
I run on GPU

In [ ]:
pip --version

In [ ]:
!git version

In [4]:
!git clone https://github.com/MIC-DKFZ/HD-BET

Cloning into 'HD-BET'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 188 (delta 38), reused 33 (delta 33), pack-reused 139
Receiving objects: 100% (188/188), 43.52 KiB | 4.83 MiB/s, done.
Resolving deltas: 100% (120/120), done.


In [5]:
cd /kaggle/working/HD-BET

/kaggle/working/HD-BET


In [6]:
pip install -e .

Obtaining file:///kaggle/working/HD-BET
  Preparing metadata (setup.py) ... done
  Running setup.py develop for HD_BET
Note: you may need to restart the kernel to use updated packages.


In [7]:
pwd

'/kaggle/working/HD-BET'

In [ ]:
!lsb_release -d

In [8]:
import os
out_folder =  '/kaggle/working/CN_FLW_BET_3'
os.makedirs(out_folder, exist_ok=True) # create folder if not exists

In [9]:
!hd-bet -i '/kaggle/working/gz_CN_TimePoints_3' -o '/kaggle/working/CN_FLW_BET_3'

/opt/conda/bin/hd-bet:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  __import__('pkg_resources').require('HD-BET==1.0')

########################
If you are using hd-bet, please cite the following paper:
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W,Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificialneural networks. arXiv preprint arXiv:1901.11341, 2019.
########################

File: /kaggle/working/gz_CN_TimePoints_3/068_S_4174_a.nii.gz
preprocessing...
image shape after preprocessing:  (171, 160, 141)
prediction (CNN id)...
0
1
2
3
4
running postprocessing... 
exporting segmentation...
File: /kaggle/working/gz_CN_TimePoints_3/068_S_4174_b.nii.gz
preprocessing...
image shape after preprocessing:  (171, 160, 141)
prediction (CNN id)...
0
1
2
3
4
running postprocessing... 
exporting segme

# FSL
implemented on my local system

In [ ]:
cd /content/drive/MyDrive/Thesis/

In [ ]:
!sudo apt -qq install file

In [ ]:
!python3 fslinstaller.py

In [ ]:
import os

In [ ]:
fslpath = "/usr/local/fsl"
os.environ["FSLDIR"] = fslpath
os.environ["PATH"] += os.pathsep + os.path.join(fslpath, 'bin')

In [ ]:
!. ${FSLDIR}/etc/fslconf/fsl.sh

In [ ]:
!flirt -version

In [ ]:
!. ${FSLDIR}/etc/fslconf/fsl.sh

In [ ]:
!flirt -in /content/drive/MyDrive/Thesis/Data/AD/AD_baseline_ss/AD_002_S_0619 -ref /content/drive/MyDrive/fsl/data/standard/MNI152_T1_1mm_brain -out /content/drive/MyDrive/Thesis/AD_01 -omat invol2refvol.mat -dof 12


In [ ]:
from IPython.display import JSON
from google.colab import output
from subprocess import getoutput
import os

def shell(command):
  if command.startswith('cd'):
    path = command.strip().split(maxsplit=1)[1]
    os.chdir(path)
    return JSON([''])
  return JSON([getoutput(command)])

In [ ]:
output.register_callback('shell', shell)

In [ ]:
!/content/drive/MyDrive/fsl/bin/flirt -in /content/drive/MyDrive/Thesis/Data/AD/AD_baseline_ss/AD_002_S_0619.nii.gz -ref /content/drive/MyDrive/fsl/data/standard/MNI152_T1_1mm_brain.nii.gz -out /content/drive/MyDrive/Thesis/Dataset/ADNI_Processed/AD/AD_01 -omat invol2refvol.mat -dof 12


# Crop
I dont use crop

In [ ]:
import nibabel as nib
import os

In [ ]:
BASE_DIR = os.path.dirname("/content/drive/MyDrive/Thesis/")
print(f'project folder = {BASE_DIR}')
# Define the directory path
directory_path = '/content/drive/MyDrive/Thesis/Data/AD/AD_baseline_cls1_PR/'
# Iterate through files in the directory and add filenames with extensions to raw_examples list

raw_examples = []

for filename in os.listdir(directory_path):
    # Check if the path refers to a file (not a subdirectory)
    if os.path.isfile(os.path.join(directory_path,filename)):
        raw_examples.append(filename)

# Display the updated raw_examples list
print(len(raw_examples))

In [ ]:
# loop through the entire dataset to do N4 bias field correction
for volume in raw_examples:
  img_path = os.path.join(BASE_DIR, 'Data', 'AD' , 'AD_baseline_cls1_PR', volume)
  vol = nib.load(img_path).get_fdata()
  cropped = vol[12:171,14:205,3:162]
  print(cropped.header.get_data_shape()) #160*192*160
  nib.save(cropped, directory_path) #save in same path


# Normalization

I added this part to the Implementation notebook

In [ ]:
import torch
import nibabel as nib
import torchvision
from torchvision.transforms.functional import normalize

In [ ]:
import glob
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
# build dataloaders with pytorch
class MRIDataset(Dataset):
  def __init__(self) -> None:
      self.data_path = '/content/drive/MyDrive/Thesis/Data/'
      file_list = glob.glob(self.data_path + '*') # this searches for ALL directories
      print(file_list)
      self.data = []
      for class_path in file_list:
        class_name = class_path.split("/")[-1]
        for vol_path in glob.glob(class_path + "/*.nii.gz"):
          self.data.append([vol_path, class_name])
      print(self.data)
      self.class_map = {'AD':0 , 'MCI':1 , 'CN' : 2}
      super().__init__()

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    vol_path, class_name = self.data[idx]

    volume = nib.load(vol_path).get_fdata()
    class_index = self.class_map[class_name]

    # to normalization
    mean = np.mean(volume)
    std = np.std(volume)

    volume = torch.tensor(volume).float() # convert to pytorch tensor
    class_index = torch.tensor(class_index)   #.float

    volume = normalize(volume, mean, std) # normalized volume

    # check mean and variance
    # print(torch.var(volume))
    # print(torch.mean(volume))

    return volume , class_index

In [ ]:
dataset = MRIDataset()
dataloader = DataLoader(dataset , batch_size= 1 , shuffle=True) # the dimensions should be same

In [ ]:
# lets take a look at data
i = 0
for imgs, labels in dataloader:
    i += 1
    print("batch No",i)
    # print(imgs)
    # print()
    # print(labels)
    print("Batch of images has shape: ",imgs.shape)
    print("Batch of labels has shape: ", labels.shape)

In [ ]:
iterable = iter(dataloader)
data = next(iterable)
data[0].shape